# 03 — Backtest report

Aggregates `OutcomeWindow` rows into per-creator summaries.

Read this with the assumptions in [`docs/decisions/0003-backtest-assumptions.md`](../docs/decisions/0003-backtest-assumptions.md):
- Primary horizon: 5 trading days, conditional on activation, net of 10 bps round-trip cost.
- All numbers shown should be read alongside `n_activated` — small N collapses everything to noise.
- Excess return = creator return − SPY return over the same wall-clock window.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sqlalchemy import select

from app.db import session_scope
from app.models import (
    Creator, Document, ExtractedCall, OutcomeWindow, SourceChannel
)

pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## Load all outcomes

In [ ]:
with session_scope() as s:
    rows = s.execute(
        select(
            Creator.display_name.label('creator'),
            Document.posted_at,
            ExtractedCall.id.label('call_id'),
            ExtractedCall.ticker,
            ExtractedCall.direction,
            ExtractedCall.entry_type,
            OutcomeWindow.horizon,
            OutcomeWindow.activated,
            OutcomeWindow.return_pct,
            OutcomeWindow.gross_return_pct,
            OutcomeWindow.benchmark_return_pct,
            OutcomeWindow.excess_return_pct,
            OutcomeWindow.mfe,
            OutcomeWindow.mae,
            OutcomeWindow.hit_target,
            OutcomeWindow.hit_stop,
            OutcomeWindow.status,
        )
        .join(SourceChannel, SourceChannel.creator_id == Creator.id)
        .join(Document, Document.source_channel_id == SourceChannel.id)
        .join(ExtractedCall, ExtractedCall.document_id == Document.id)
        .join(OutcomeWindow, OutcomeWindow.call_id == ExtractedCall.id)
    ).all()

df = pd.DataFrame(rows)
print(f'{len(df)} outcome rows')
df.head(15)

## Per-creator summary at the primary horizon (5d)

In [ ]:
primary = df[df['horizon'] == '5d'].copy()
by_creator = primary.groupby('creator').agg(
    n_calls=('call_id', 'nunique'),
    n_activated=('activated', 'sum'),
    hit_rate=('return_pct', lambda s: (s.dropna() > 0).mean()),
    mean_return=('return_pct', 'mean'),
    median_return=('return_pct', 'median'),
    mean_excess=('excess_return_pct', 'mean'),
    mean_mfe=('mfe', 'mean'),
    mean_mae=('mae', 'mean'),
)
by_creator['activation_rate'] = by_creator['n_activated'] / by_creator['n_calls']
by_creator.sort_values('mean_excess', ascending=False)

## Per-horizon view

How does mean excess-vs-SPY change as the horizon lengthens?

In [ ]:
by_horizon = df.groupby(['creator', 'horizon']).agg(
    n=('call_id', 'count'),
    activated=('activated', 'sum'),
    mean_excess=('excess_return_pct', 'mean'),
    mean_return=('return_pct', 'mean'),
)
by_horizon

## Activated calls — full detail

In [ ]:
activated = df[df['activated'] & (df['horizon'] == '5d')].sort_values('return_pct', ascending=False)
activated[['creator', 'ticker', 'direction', 'entry_type', 'return_pct', 'excess_return_pct', 'mfe', 'mae']]

## Honest reading

- These are bootstrap numbers from a single small backfill. **N is too small to draw any conclusion** about creator skill.
- Differences between creators here are dominated by which specific tickers they happened to call in the sampled period — not by any consistent edge.
- Re-evaluate this notebook after at least one full quarter of accumulated activated calls per creator (target: ≥30 per creator before treating numbers as inferential).
- Look at `mean_excess` — that's the creator's return *over and above* what SPY returned in the same window. Raw `mean_return` flatters or punishes creators based on regime, not skill.